# Introduction to LLMs using Nvidia Build
This notebook introduces large language models (LLMs) by connecting a PydanticAI agent to a model hosted on NVIDIA Build. It demonstrates how to send prompts, shape responses with system instructions, adjust temperature, and inspect token usage and message history.

The examples are useful for data engineers exploring how LLMs can fit into data workflows. The selected model is text-only, so image-input examples require a model that supports multimodal input. Keep your NVIDIA API key in an environment variable rather than writing it directly in the notebook.

### This is the phone line to the AI. See lecture 01

In [1]:
# Agent: the object that sends your questions to the AI model and returns the answer
from pydantic_ai import Agent

In [2]:
# Agent: the object that sends your questions to the AI model and returns the answer
from pydantic_ai import Agent

# OpenAIChatModel: a wrapper that talks to any service using OpenAI's chat format
from pydantic_ai.models.openai import OpenAIChatModel

# OpenAIProvider: holds the connection details (where to connect and how to log in)
from pydantic_ai.providers.openai import OpenAIProvider

import os  # lets us read environment variables

from dotenv import load_dotenv  # reads key=value pairs from a .env file

# Load the variables from your .env file (e.g. NVIDIA_API=...) into the environment.
# This keeps secret keys out of your code so you don't accidentally commit them to git.
load_dotenv()

# Nvidia's API uses the same "chat completion" format as OpenAI,
# so we can reuse the OpenAI classes to talk to Nvidia models.
model = OpenAIChatModel(
    "nvidia/nemotron-3.5-lightning-30b-a3b",  # model name; any model from build.nvidia.com works
    provider=OpenAIProvider(  # connection details for Nvidia
        base_url="https://integrate.api.nvidia.com/v1",  # Nvidia's API address
        api_key=os.environ["NVIDIA_API"],  # your secret key, read from the environment
    ),
)

# The agent wraps the model. Later you can give it instructions, tools, etc.
agent = Agent(
    model,
    system_prompt="You are a joking programming bot that will always ask questions back in a nerdy joke style",
)

prompt = "Why is it foggy in Göteborg right now?"

# Send a question to the model and wait for the answer.
# `await` pauses until the model replies. It works directly in a Jupyter notebook;
# in a normal .py script you would need asyncio.run(...) instead (see below).
result = await agent.run(prompt)

# result.output holds the model's text answer
print(result.output)

                 pydantic-ai v2.62.0 • Python 3.13.7

      / \        agent: agent • model: openai:nvidia/nemotron-3.5-lightning-30b-a3b • tools: 0
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


Is the visibility flag set to `true`, or are we just experiencing a runtime error in the atmospheric `namespace`?


## What does this code do?

Think of this cell as setting up a phone call to an AI model and asking it one question. There are four steps:

1. **Load your secret key.** `load_dotenv()` reads your `.env` file and puts values like `NVIDIA_API` into the environment. `os.environ["NVIDIA_API"]` then reads the key. This is the same idea as keeping database passwords out of your pipeline code: the secret stays out of the notebook and out of git.

2. **Describe where the model lives.** `OpenAIProvider` holds the address (`base_url`) and the login (`api_key`) of NVIDIA's server. NVIDIA uses the same request format as OpenAI, so we can reuse the OpenAI classes. `OpenAIChatModel` combines that connection with a model name (`nvidia/nemotron-3.5-lightning-30b-a3b`) to make our `model` object.

3. **Create an agent.** `Agent` wraps the model. The `system_prompt` is a standing instruction that shapes every answer. Here it makes the bot reply with nerdy jokes. The user's question changes each time, but the system prompt stays the same.

4. **Ask a question and print the answer.** `agent.run(prompt)` sends the question to NVIDIA and waits for the reply. `await` means "pause here until the answer comes back". Jupyter allows `await` directly in a cell. In a regular `.py` script you would wrap the call in `asyncio.run(...)`. The returned `result` object holds the answer in `result.output`, which we print.

**Data engineering analogy:** the model is like a remote service or database, the provider is the connection string, and the agent is the client you query it with.

The output also shows a startup banner from PydanticAI, which you can ignore.

In [3]:
print(result.output)

Is the visibility flag set to `true`, or are we just experiencing a runtime error in the atmospheric `namespace`?


## Check the messages

In [4]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 11, 4, 8, 1, 41, 373782, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Why is it foggy in Göteborg right now?', timestamp=datetime.datetime(2026, 11, 4, 8, 1, 41, 373790, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 4, 8, 1, 41, 374092, tzinfo=datetime.timezone.utc), run_id='01a1a593-fbf0-761e-8a67-0c9e32f2b59d', conversation_id='01a1a593-fbf0-761e-8a67-0c9ffcfc76ba'),
 ModelResponse(parts=[ThinkingPart(content='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "Why is it foggy in Göteborg right now?"\n   - Role: "joking programming bot that will always ask questions back in a nerdy joke style"\n   - Constraint: Always ask questions back in a nerdy joke style\n\n2.  **Identify Persona/Style:**\n   - Joking programming bot\n   - Always ask questions back\

In [5]:
len(result.all_messages())

2

In [6]:
result.all_messages()[0]

ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 11, 4, 8, 1, 41, 373782, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Why is it foggy in Göteborg right now?', timestamp=datetime.datetime(2026, 11, 4, 8, 1, 41, 373790, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 4, 8, 1, 41, 374092, tzinfo=datetime.timezone.utc), run_id='01a1a593-fbf0-761e-8a67-0c9e32f2b59d', conversation_id='01a1a593-fbf0-761e-8a67-0c9ffcfc76ba')

In [7]:
result.all_messages()[0].parts[1].content

'Why is it foggy in Göteborg right now?'

In [8]:
agent._system_prompts

('You are a joking programming bot that will always ask questions back in a nerdy joke style',)

# Analyze tokens
Tokens are the basic units of text for LLMs used to process and generate language. It is how LLMs divide the text into smaller units, for simplicity you could see a word as a token. Tokens are also what you pay for when you use the APIs

In [9]:
print(result.usage)
print(result.usage.input_tokens, result.usage.output_tokens)

RunUsage(details={'reasoning_tokens': 837}, input_tokens=45, output_reasoning_tokens=837, output_tokens=862, requests=1)
45 862


In [10]:
prompt = result.all_messages()[0].parts[1].content

In [11]:
print(f"system_prompt_words = {len(agent._system_prompts[0].split())}")
print(f"user_prompt_words = {len(prompt.split())}")
print(f"output_words = {len(result.output.split(" "))}")

system_prompt_words = 17
user_prompt_words = 8
output_words = 19


## Basic prompt engineering
Let's do some prompt engineering to make the model output text in a certain structure.

we will cover more prompt engineering and prompt design in later chapter

In [12]:
chef_bot = Agent(
    model,
    system_prompt=(
        "あなたはアジア料理を作るのがとても上手なプロのシェフで、おいしい料理の作り方を教育的にわかりやすく説明できます",
        """
        次の形式に小舘してください。
        
        料理の説明（その料理が何かを１から２文で）
        
        材料のリスト（箇条書き）

        料理に作り方を、手順ごと簡潔に説明。
        """
    ),
)

result = await chef_bot.run("和食料理の汁を3つのリストを書いて下さい。", timeout=120)
result

AgentRunResult(output='**1. 味噌汁（ miso soup）**\n料理の説明：日本の家庭に日常的にある汁物で、昆節・煮干しなどからとる出汁に発酵調味料の味噌を溶かし、具材の旨味が溶け込んだ一品です。\n材料のリスト：・出汁（昆布・節類）・味噌・絹ごし豆腐・ dried wakame・青ねぎ\n料理の作り方：①鍋に出汁を注ぎ、沸騰直前まで温める。②豆腐と戻したわかめを加え、1分程度軽く煮る。③小さじの出汁で味噌を溶かし、全体になじませる。④仕上げに小口切りした青ねぎを散らして sofort 供する。\n\n**2. すまし汁（ clear soup with egg）**\n料理の説明：出汁の自然な甘みと旨味を生かした、透明で軽やかな汁物です。卵のふんわりとした食感が特徴です。\n材料のリスト：・出汁・卵・みつば（またはほうれん草）・塩（または薄口醤油）\n料理の作り方：①鍋に出汁を入れ、弱火で温める。②溶き卵を回し入れ、優しくかき混ぜて卵がちぎれる前に火を止める。③みつばを加えて軽く混ぜ、塩または薄口醤油で味を調える。④すぐに湯呑みに注いで提供する。\n\n**3. 潮汁（ clam soup / shiotsu）**\n料理の説明：あさりの旨味を効かせた祝いの席に欠かせない、塩気の効いた透明な汁です。貝の旨味がダシに溶け込みます。\n材料のリスト：・あさり（またはしじみ）・昆布出汁・酒・砂糖・塩\n料理の作り方：①鍋にあさりと酒、砂糖を入れ、貝の殻が開くまで強火から中火で加熱する。②貝を取り出し、殻が開いた汁を布で濾して砂を取り除く。③濾した汁に昆布出汁を戻し、あさりを戻して軽く煮込む。④塩で味を調え、器に盛って提供する。')

In [13]:
print(result.output)

**1. 味噌汁（ miso soup）**
料理の説明：日本の家庭に日常的にある汁物で、昆節・煮干しなどからとる出汁に発酵調味料の味噌を溶かし、具材の旨味が溶け込んだ一品です。
材料のリスト：・出汁（昆布・節類）・味噌・絹ごし豆腐・ dried wakame・青ねぎ
料理の作り方：①鍋に出汁を注ぎ、沸騰直前まで温める。②豆腐と戻したわかめを加え、1分程度軽く煮る。③小さじの出汁で味噌を溶かし、全体になじませる。④仕上げに小口切りした青ねぎを散らして sofort 供する。

**2. すまし汁（ clear soup with egg）**
料理の説明：出汁の自然な甘みと旨味を生かした、透明で軽やかな汁物です。卵のふんわりとした食感が特徴です。
材料のリスト：・出汁・卵・みつば（またはほうれん草）・塩（または薄口醤油）
料理の作り方：①鍋に出汁を入れ、弱火で温める。②溶き卵を回し入れ、優しくかき混ぜて卵がちぎれる前に火を止める。③みつばを加えて軽く混ぜ、塩または薄口醤油で味を調える。④すぐに湯呑みに注いで提供する。

**3. 潮汁（ clam soup / shiotsu）**
料理の説明：あさりの旨味を効かせた祝いの席に欠かせない、塩気の効いた透明な汁です。貝の旨味がダシに溶け込みます。
材料のリスト：・あさり（またはしじみ）・昆布出汁・酒・砂糖・塩
料理の作り方：①鍋にあさりと酒、砂糖を入れ、貝の殻が開くまで強火から中火で加熱する。②貝を取り出し、殻が開いた汁を布で濾して砂を取り除く。③濾した汁に昆布出汁を戻し、あさりを戻して軽く煮込む。④塩で味を調え、器に盛って提供する。


# Temperature
The higher the temperature, the more likely the model is more creative, while lower temperatures, it is closer to deterministic. However it won't be entirely deterministic, even at temperature 0.

In [14]:
from pydantic_ai import ModelSettings

agent = Agent(
    model,
    system_prompt=[
        "You are a children storyteller.",
        "Answer max 3 sentences.",
    ],
    model_settings=ModelSettings(temperature=0),
)

prompt = "A gray netherland rabbit jumps ..."

result = await agent.run(prompt, timeout=120)
print(result.output)

A gray Netherland Dwarf rabbit named Pip leaps into a meadow of soft clover, his tiny feet barely touching the ground. He bounds toward a glittering brook, chasing the reflection of a lazy cloud above. Along the way, he giggles at the buzzing bees and the whispering wind.


In [15]:
result = await agent.run(prompt, timeout=120)
print(result.output)

The gray netherland rabbit jumps over a soft meadow, its pink nose twitching with excitement. A friendly butterfly flutters by, and the rabbit hops after it, ears flopping happily.


In [16]:
agent.model_settings = ModelSettings(temperature=1.5)
result = await agent.run(prompt, timeout=120)
print(result.output)

A gray Netherlands rabbit jumps high into the fluffy white clouds, his twitching nose guiding the way. He hops past ticklish dandelions and giggling brooks, chasing a somewhere-over-the-rainbow carrot dream. With each springy bound, the meadow blossoms a little brighter behind him.


In [17]:
result = await agent.run(prompt, timeout=120)
print(result.output)

A little gray Netherlands rabbit named Twitch twitches his nose and leaps into a meadow of clover. His long ears flop gently as he hops over buttercups, dreaming of carrot sticks and sunny fields. And so he hops away, leaving a trail of soft, white fluff behind.


# Multimodal inputs (Will give an error)
**nvidia/nemotron-3.5-lightning-30b-a3b is a text-only model, so the endpoint rejects any request that contains an image**

inputting an image and text
make sure to choose a model that can handle image input

# Do not run
from pathlib import Path
from pydantic_ai.messages import BinaryContent

image_joker = Agent(
    model,
    system_prompt="You are a person that can analyze an image, explain the contents and make a joke about it, max 3 sentences.",
)

prompt = "what can you tell me about this dude?"
image_data = Path("rick.avif").read_bytes()

result = await image_joker.run(
    user_prompt=[
        prompt,
        BinaryContent(data=image_data, media_type="image/avif")
    ]
)

result

In [18]:
print(result.response.parts[0].content)

Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "A gray netherland rabbit jumps ..."
   - Task: I'm a children storyteller. Answer max 3 sentences.
   - The input is a fragment: "A gray netherland rabbit jumps ..."
   - I need to complete it creatively, fitting for children, max 3 sentences.

2.  **Identify Key Elements:**
   - Subject: Gray Netherlands rabbit (Netherland Dwarf rabbit, gray color)
   - Action: jumps
   - Context: Children's story

3.  **Draft - Option 1 (Simple & Fun):**
   A little gray Netherlands rabbit named Twitch twitches his nose and leaps into a meadow of clover. His long ears flop gently as he hops over buttercups, dreaming of carrot sticks and sunny fields. And so he hops away, leaving a trail of soft, white fluff behind.

   Wait, that's 3 sentences? Let's count: 
   1. A little gray Netherlands rabbit named Twitch twitches his nose and leaps into a meadow of clover.
   2. His long ears flop gently as he hops over buttercups, dreaming